# Truy xuất đoạn văn tiếng Việt bằng BM25

Dành cho thành viên xây dựng hệ thống hỏi đáp ViMMRC. Cần Python 3.10+; chỉ dùng thư viện chuẩn. Sau bài này, bạn sẽ tạo corpus `{id, passage}`, truy xuất ngữ cảnh dạng chuỗi, lưu/tải index và đo Recall@k/MRR@k. Toàn bộ ví dụ dưới đây là **dữ liệu tổng hợp**, không phải ViMMRC.


## Lộ trình

1. Nạp ví dụ và tạo BM25 index.
2. Tìm top-k và lấy chuỗi cho MRC.
3. Lưu/tải index và đánh giá bằng ID bài đọc.
4. So cấu hình cũ với cấu hình đã đo lường.
5. Tự thử câu hỏi khác; xem giới hạn của BM25.


## 1. Chuẩn bị

Chạy notebook từ thư mục gốc repository hoặc từ `retrieval-model_Huy/`. Cell này xác định đường dẫn module và nạp corpus nhỏ đi kèm.


In [ ]:
import json
from pathlib import Path
import sys

module_dir = Path.cwd()
if not (module_dir / "retrieval_client.py").exists():
    module_dir = module_dir / "retrieval-model_Huy"
assert (module_dir / "retrieval_client.py").exists(), "Hãy chạy từ gốc repository hoặc retrieval-model_Huy/"
sys.path.insert(0, str(module_dir))

from retrieval_client import Retriever
from retrieval_cli import evaluate

passages = json.loads((module_dir / "synthetic_passages.json").read_text(encoding="utf-8"))
[(item["id"], item["passage"]) for item in passages]


## 2. Tìm đoạn liên quan

BM25 đếm từ trùng giữa câu hỏi và mỗi passage, điều chỉnh theo độ dài và độ phổ biến của từ. `search()` trả ID, passage và điểm; điểm chỉ dùng để **xếp hạng trong cùng index**.


In [ ]:
retriever = Retriever(passages)
question = "Hành tinh nào gần Mặt Trời nhất?"
hits = retriever.search(question, top_k=3)
[(hit["id"], round(hit["score"], 3)) for hit in hits]


Đoạn `mercury` cần đứng đầu. `retrieve()` trả văn bản thuần để chuyển thẳng cho MRC. Khi không có từ trùng, nó trả chuỗi rỗng; ứng dụng nên xử lý trường hợp này.


In [ ]:
context = retriever.retrieve(question)
assert context == passages[1]["passage"]
print(context)
print("Không tìm thấy:", repr(retriever.retrieve("không-trùng-khớp")))


## 3. Lưu và tải index

JSON chỉ chứa corpus và tham số BM25. Lưu vào thư mục tạm để notebook không tạo file phụ trong repository.


In [ ]:
import tempfile

with tempfile.TemporaryDirectory() as temporary_dir:
    index_path = Path(temporary_dir) / "index.json"
    retriever.save(index_path)
    reloaded = Retriever.load(index_path)
    assert reloaded.retrieve(question) == context
print("Tải lại index thành công")


## 4. Đánh giá truy xuất

Mỗi ví dụ đánh giá có `question` và `relevant_ids` (ID passage đúng). Recall@k là tỷ lệ passage đúng trong top-k; MRR@k phản ánh vị trí xuất hiện đầu tiên. Không đưa đáp án trắc nghiệm vào index hay dùng nó làm nhãn truy xuất.


In [ ]:
queries = json.loads((module_dir / "synthetic_queries.json").read_text(encoding="utf-8"))
metrics = evaluate(retriever, queries, top_k=1)
metrics


## 5. Cấu hình đã được đo lường

Mặc định dùng BM25 đã chỉnh trên 800 câu hỏi train: bỏ các từ hỏi quá phổ biến khỏi kênh unigram và thưởng nhẹ cho cặp từ đứng cạnh nhau. Có thể dựng cấu hình BM25 cũ bằng `BASELINE_CONFIG` để so sánh trên cùng corpus. Chỉ số tổng hợp nằm trong báo cáo, không có văn bản bài đọc.


In [ ]:
from retrieval_client import BASELINE_CONFIG, OPTIMIZED_CONFIG

baseline = Retriever(passages, **BASELINE_CONFIG)
print("BM25 cũ:", BASELINE_CONFIG)
print("BM25 hiện tại:", OPTIMIZED_CONFIG)
print("Đoạn đứng đầu:", baseline.search(question, 1)[0]["id"], retriever.search(question, 1)[0]["id"])


Báo cáo dưới đây cho thấy train được dùng để chọn tham số, sau đó mới đo một lần trên dev. Đây vẫn là phép truy xuất trong corpus chứa bài đọc gốc; không đại diện cho khả năng tìm bài đọc vắng khỏi index.


In [ ]:
report = json.loads((module_dir / "reports" / "lexical_benchmark.json").read_text(encoding="utf-8"))
{"train_winner_config": report["train_winner_config"],
 "dev_baseline_recall@1": report["dev_baseline"]["recall@1"],
 "dev_tuned_recall@1": report["dev_winner"]["recall@1"]}


## Bài tập

Thử câu hỏi về sự hình thành mưa. Hãy đoán ID đầu tiên rồi chạy cell. Với câu hỏi chỉ dùng từ đồng nghĩa khác hẳn corpus, BM25 có thể không tìm được đoạn nào; muốn xử lý trường hợp đó có thể thử embedding retrieval về sau.


In [ ]:
rain_question = "Mưa hình thành từ điều gì?"
rain_hit = retriever.search(rain_question, top_k=1)
assert rain_hit[0]["id"] == "rain"  # đáp án kiểm tra
rain_hit[0]["passage"]


## Áp dụng với dữ liệu thật

Nếu đã có `data/dev_vimmrc.json`, chạy từ gốc repository:

```powershell
python retrieval-model_Huy/retrieval_cli.py prepare-vimmrc --input data/dev_vimmrc.json --output retrieval-model_Huy/dev_index.json --split dev
python retrieval-model_Huy/retrieval_cli.py evaluate-vimmrc --index retrieval-model_Huy/dev_index.json --data data/dev_vimmrc.json --split dev --top-k 5
```

Đây là phép đo **closed corpus** vì bài đọc dev đã nằm trong index. Khi đánh giá câu hỏi dev trên index train, xem thêm `corpus_coverage`: bài đọc dev vắng khỏi corpus thì không thể truy xuất đúng ID. Xem `README.md` để biết lệnh đầy đủ.
